# R4 — Policy-free gold orders_enriched

Builds `hop.gold.orders_enriched`: every clean order joined to customer, zone and the restaurant version in force at `placed_ts` (SCD2 temporal join), with unrounded DOUBLE timing measures. **Contract: facts only** — no is_late/late/eligible/fault/compensation/amount/reason_code columns. The transformation code runs in the **hop_pipeline** SDP (`src/hop_pipeline/transformations/orders_enriched.py`); this notebook documents the round and verifies its outputs.

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
print(CATALOG, CHAT_MODEL, EMBED_MODEL)

## What R4 builds (in the hop_pipeline SDP)

* 25 contract columns: keys, degenerate dims, timestamps, `placed_to_accepted_minutes`, `prep_minutes`, `actual_minutes`, `minutes_past_promise` (all DOUBLE, measured from accepted, unrounded), plus customer city/tier, zone type, and restaurant name/zone/rating **at order time**
* Times and amounts stay policy-free: any late/compensation decision is made downstream, never in gold

In [0]:
# R4 — verification (read-only): contract shape, no policy columns, temporal join
g = spark.table("hop.gold.orders_enriched")
cols = g.columns
print("rows:", g.count(), "| columns:", len(cols), "(expect 25)")

policy = [c for c in cols if c.lower() in (
    "is_late", "late", "late_threshold_minutes", "eligible", "is_eligible", "fault",
    "compensation", "compensation_type", "coupon_amount", "refund_amount", "amount", "reason_code")]
print("policy columns present (expect []):", policy)

# temporal join: every delivered order must carry the restaurant version in force at placed_ts
print("delivered orders without restaurant version:",
      g.filter("status = 'delivered' AND restaurant_name IS NULL").count())
display(g.filter("order_id = 'ORD-42809'").select(
    "order_id", "restaurant_id", "restaurant_rating_at_order", "placed_ts",
    "actual_minutes", "minutes_past_promise"))